# Feature Track 3: Query Improvement Methods

Standard RAG embeds the user query as-is and retrieves the nearest chunks. This works when the query is well-formed, but a single phrasing can miss relevant chunks whose wording differs.

This notebook covers two query-side techniques that improve retrieval without changing the vector store or the LLM:

| Technique | Idea |
|---|---|
| **Query Expansion** | Generate N alternative phrasings of the query; retrieve for each and merge results via RRF |
| **HyDE** | Ask the LLM to draft a hypothetical answer first; embed that answer instead of the raw query |

---

## Setup

**Prerequisites:** `conversational_toolkit` and `backend` must be installed in editable mode. Set `OPENAI_API_KEY` for OpenAI.

In [ ]:
from loguru import logger
import sys

from conversational_toolkit.embeddings.openai import OpenAIEmbeddings
from conversational_toolkit.agents.base import QueryWithContext
from conversational_toolkit.vectorstores.chromadb import ChromaDBVectorStore

from sme_kt_zh_collaboration_rag.feature0_baseline_rag import (
    load_chunks,
    build_vector_store,
    build_llm,
    build_agent,
    SYSTEM_PROMPT,
    EMBEDDING_MODEL,
    VS_PATH,
)

logger.remove()
logger.add(sys.stderr, level="WARNING")

# Choose your LLM backend (only needed for the final answer cells)
BACKEND = "openai"  # "ollama", "openai", or "qwen"
FORCE_REBUILD = False  # set True to re-embed from scratch and rebuild the vector store

embedding_model = OpenAIEmbeddings(model_name=EMBEDDING_MODEL)
print(f"Embedding model: {EMBEDDING_MODEL}")

if FORCE_REBUILD or not VS_PATH.exists():
    chunks = load_chunks(max_files=5)
    chunks = [c for c in chunks if c.mime_type.startswith("text")]
    vector_store = await build_vector_store(
        chunks, embedding_model, db_path=VS_PATH, reset=FORCE_REBUILD
    )
else:
    # Vector store already exists -> open it without re-embedding
    vector_store = ChromaDBVectorStore(db_path=str(VS_PATH))
    print(
        f"Reusing existing vector store at {VS_PATH} ({vector_store.collection.count()} chunks)"
    )

llm = build_llm(backend=BACKEND)
print(f"LLM backend: {BACKEND}")

### Baseline (no expansion, `number_query_expansion=0`)

In [ ]:
agent = build_agent(
    vector_store=vector_store,
    embedding_model=embedding_model,
    llm=llm,
    top_k=5,
    system_prompt=SYSTEM_PROMPT,
    number_query_expansion=0,
)

In [ ]:
query = "Which pallets in our portfolio have a third-party verified EPD?"
query_with_context = QueryWithContext(query=query, history=[])

response = await agent.answer(query_with_context)

print("\nAnswer:")
print(f"{response.content[0].text}")
print(f"\nSources ({len(response.sources)}):")
for src in response.sources:
    source_file = src.metadata.get("source_file", "?")
    print(f"  {source_file!r}  |  {src.title!r}")

---

## Query Expansion

A single query phrasing may not match the wording of the most relevant chunks. Query expansion asks an LLM to generate `N` alternative phrasings of the original query. Each phrasing is embedded and retrieved independently, then all result lists are merged using **Reciprocal Rank Fusion (RRF)**, the same rank-based merge used in hybrid retrieval.

```
Original query ──► embed ──► top-k list 1 ─┐
Alt. query 1   ──► embed ──► top-k list 2 ─┤─► RRF merge ──► final top-k
...                                        │
Alt. query N   ──► embed ──► top-k list N ─┘
```

**When it helps:** queries that can be expressed in multiple ways, or where the user's phrasing is unlikely to closely match document wording.  
**Cost:** one extra LLM call + N extra embedding calls per query.

The DEBUG output shows the 5 generated phrasings before retrieval. Each is embedded separately and the results are merged via RRF.

In [ ]:
logger.add(sys.stderr, level="DEBUG")

agent_query_expansion = build_agent(
    vector_store=vector_store,
    embedding_model=embedding_model,
    llm=llm,
    top_k=5,
    system_prompt=SYSTEM_PROMPT,
    number_query_expansion=5,
)

In [ ]:
query = "Which pallets in our portfolio have a third-party verified EPD?"
query_with_context = QueryWithContext(query=query, history=[])

response = await agent_query_expansion.answer(query_with_context)

print("\nAnswer:")
print(f"{response.content[0].text}")
print(f"\nSources ({len(response.sources)}):")
for src in response.sources:
    source_file = src.metadata.get("source_file", "?")
    print(f"  {source_file!r} | {src.title!r}")

---

## HyDE: Hypothetical Document Embeddings

The core insight is that embedding a *question* and embedding a *document* produce vectors in different regions of the embedding space, even when the question and document are semantically aligned. HyDE closes this gap:

1. The original query is sent to an LLM, which drafts a **hypothetical answer** without access to the vector store (pure hallucination is fine here)
2. That hypothetical answer is embedded instead of the raw query
3. The embedding of a document-like answer is closer to real document embeddings -> better retrieval

```
Query ──► LLM (no context) ──► hypothetical answer ──► embed ──► top-k
```

**When it helps:** open-ended questions where the query phrasing is far from how documents are written (e.g. "tell me about..." vs. declarative document text).  
**Risk:** if the hypothetical answer hallucinates a wrong domain, retrieval can be misled.  
**Cost:** one extra LLM call per query (same model as the main LLM).

### Baseline (no HyDE, `enable_hyde=False`)

In [ ]:
query = "Which pallets in our portfolio have a third-party verified EPD?"
query_with_context = QueryWithContext(query=query, history=[])

response = await agent.answer(query_with_context)

print("\nAnswer:")
print(f"{response.content[0].text}")
print(f"\nSources ({len(response.sources)}):")
for src in response.sources:
    source_file = src.metadata.get("source_file", "?")
    print(f"  {source_file!r} | {src.title!r}")

### With HyDE (`enable_hyde=True`)

The DEBUG output shows the hypothetical answer generated before retrieval. That text, not the original query, is embedded to find the nearest chunks.

In [ ]:
agent_hyde = build_agent(
    vector_store=vector_store,
    embedding_model=embedding_model,
    llm=llm,
    top_k=5,
    system_prompt=SYSTEM_PROMPT,
    number_query_expansion=0,
    enable_hyde=True,
)

In [ ]:
query = "Which pallets in our portfolio have a third-party verified EPD?"
query_with_context = QueryWithContext(query=query, history=[])

response = await agent_hyde.answer(query_with_context)

print("Answer:")
print(f"{response.content}")
print(f"Sources ({len(response.sources)}):")
for src in response.sources:
    source_file = src.metadata.get("source_file", "?")
    print(f"  {source_file!r}  |  {src.title!r}")

---------------------------